# 7b단계: 선수 위치를 "선 기준 야드"로 바꾸기

실제 거리(m)는 쓰지 않아요. **그려진 선에 붙인 야드 값**이 기준이에요.
선과 선 사이는 비율로 계산해요 (예: 10야드 선과 20야드 선 사이 60% 지점 = 16야드).

| 순서 | 내용 |
|---|---|
| ① | 7a에서 찾은 선 + 직접 붙인 라벨 불러오기 |
| ② | 참조 화면마다 보정: 야드 선 × 폭 선(사이드·해쉬)의 교차점으로 호모그래피 → 선 사이 비율 보정 |
| ③ | 검증: 보정으로 다시 그린 선이 실제 흰 선과 겹치는지, 위에서 본 그림으로 폈을 때 선이 반듯한지 |
| ④ | v7 결과(클립 전체): ROI 갱신 프레임마다 참조와 SIFT로 맞춤 → 매 프레임 선수 발 위치를 야드로 |
| ⑤ | 활용 1: 득점자 경로를 야드로 · 활용 2: 경기장 밖(사이드라인) 인원 걸러내기 |

**런타임: CPU로 충분** (④가 클립당 1~2분)

In [ ]:
import os, json
from collections import Counter
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
drive.mount('/content/drive')

import subprocess, matplotlib.font_manager as fm
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null", shell=True)
if os.path.exists(_font):
    fm.fontManager.addfont(_font); plt.rcParams["font.family"] = "NanumGothic"
else:
    print("⚠️ 한글 글꼴 설치 실패: 그래프의 한글이 네모로 보일 수 있어요")
plt.rcParams["axes.unicode_minus"] = False

FOLDER_PATH = "/content/drive/MyDrive/2025년 홍익 vs 국민 드론뷰"
DRIVE_OUTPUT_DIR = os.path.join(FOLDER_PATH, "pipeline_outputs")
EVAL_DIR = os.path.join(DRIVE_OUTPUT_DIR, "eval")

# 참조 화면 4장 (run_pipeline · 7a와 같음): (프레임, 경기장 ROI)
KNOWN = {
    "DJI_20250913133612_0144_D": (413, [[245, 530], [1860, 543], [1919, 776], [1919, 1079], [107, 1079]]),
    "DJI_20250913135020_0156_D": (167, [[0, 0], [1632, 0], [1712, 810], [0, 787]]),
    "DJI_20250913133530_0143_D": (1, [[270, 459], [1880, 469], [1919, 612], [1919, 1079], [119, 1079]]),
    "DJI_20250913135111_0157_D": (1, [[0, 0], [1675, 0], [1761, 737], [0, 733]]),
}


def video_path_of(stem):
    for ext in (".MP4", ".mp4", ".MOV", ".mov"):
        p = os.path.join(FOLDER_PATH, stem + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"영상을 찾을 수 없습니다: {stem}")


def read_frames_sequential(video_path, frame_ids):
    """처음부터 순서대로 읽어 원하는 프레임(1부터)만 모음 (v3와 같은 프레임 번호)"""
    wanted, out, fid = set(frame_ids), {}, 0
    cap = cv2.VideoCapture(video_path)
    while fid < max(wanted):
        ok, frame = cap.read()
        if not ok:
            break
        fid += 1
        if fid in wanted:
            out[fid] = frame
    cap.release()
    return out

In [ ]:
# ===== 자동 ROI: 참조 프레임 → 새 프레임으로 경기장 영역 옮기기 =====
# 같은 경기장이면 '경기장 영역을 이미 아는 프레임(참조)'과 새 프레임은 같은 평면(경기장)을 다른 위치·각도에서 본 것.
# 평면 → 평면의 원근 변환은 3×3 호모그래피 H 하나로 표현됨:  [x', y', w']ᵀ = H [x, y, 1]ᵀ,  (x'/w', y'/w')
#   1) SIFT 특징점 추출 (크기·회전·밝기 변화에 강함) — 반 해상도에서 계산해 속도 확보
#   2) 참조 ↔ 새 프레임 특징점 짝짓기 + Lowe 비율 검사(0.75)로 애매한 짝 제거
#   3) RANSAC으로 호모그래피 추정: 무작위 4쌍으로 H를 만들고, 5px 안에 맞는 짝(인라이어)이 가장 많은 H 선택
#      → 나무·자동차처럼 경기장 평면이 아닌 점, 잘못된 짝은 자동으로 버려짐
#   4) 참조 ROI 다각형을 H로 옮김. 참조 화면 가장자리에서 잘린 변은 실제 라인 방향으로 화면 밖까지 연장한 뒤 옮김
#      (새 프레임이 더 넓게 찍혔을 때 ROI가 참조 화면 크기에서 잘리지 않도록)
#   5) 인라이어가 MIN_INLIERS 미만이면 '경기장의 다른 쪽을 찍은 영상'으로 보고 실패 처리 (수동 입력 필요)
import time
SIFT = cv2.SIFT_create(nfeatures=6000)
BF = cv2.BFMatcher()
FEAT_SCALE = 0.5
MIN_INLIERS = 100
RATIO = 0.75


def sift_features(img):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, None, fx=FEAT_SCALE, fy=FEAT_SCALE, interpolation=cv2.INTER_AREA)
    kp, des = SIFT.detectAndCompute(g, None)
    return np.float32([k.pt for k in kp]).reshape(-1, 2) / FEAT_SCALE, des


def _border_sides(p, W, H, tol=3):
    x, y = p
    return {s for s, c in (("L", x <= tol), ("R", x >= W - 1 - tol), ("T", y <= tol), ("B", y >= H - 1 - tol)) if c}


def open_polygon(poly, W, H, L=2500):
    """화면 가장자리에서 잘린 ROI 꼭짓점을 실제 라인 방향으로 L px 연장 (화면 모서리 꼭짓점은 버림)"""
    P = [np.float64(p) for p in poly]; out = []
    for i, p in enumerate(P):
        a, b = P[i - 1], P[(i + 1) % len(P)]
        bp = _border_sides(p, W, H)
        if not bp:
            out.append(p); continue
        ea, eb = bool(bp & _border_sides(a, W, H)), bool(bp & _border_sides(b, W, H))   # 이웃과 같은 가장자리 = 화면 경계 변
        if ea and eb:
            continue
        q = b if ea else a                                                              # 실제 라인 쪽 이웃
        d = (p - q) / (np.linalg.norm(p - q) + 1e-9)
        out.append(p + d * L)
    return np.array(out)


def make_ref(name, img, roi_pts):
    H, W = img.shape[:2]
    pts, des = sift_features(img)
    return {"name": name, "pts": pts, "des": des, "poly_open": open_polygon(roi_pts, W, H), "roi": np.asarray(roi_pts)}


def estimate_transform(ref, pts, des, model="homography"):
    """참조 → 새 프레임 변환과 인라이어 수. model: 'homography'(원근, 8자유도) / 'similarity'(이동·회전·크기, 4자유도)"""
    if des is None or ref["des"] is None or len(des) < 10:
        return None, 0
    good = [p[0] for p in BF.knnMatch(ref["des"], des, k=2) if len(p) == 2 and p[0].distance < RATIO * p[1].distance]
    if len(good) < 10:
        return None, 0
    a = ref["pts"][[m.queryIdx for m in good]]; b = pts[[m.trainIdx for m in good]]
    if model == "homography":
        Hm, inl = cv2.findHomography(a, b, cv2.RANSAC, 5.0)
    else:
        A, inl = cv2.estimateAffinePartial2D(a, b, method=cv2.RANSAC, ransacReprojThreshold=5.0)
        Hm = None if A is None else np.vstack([A, [0, 0, 1]])
    return (None, 0) if Hm is None else (Hm, int(inl.sum()))


def warp_roi_mask(ref, Hm, W, H):
    pts = np.hstack([ref["poly_open"], np.ones((len(ref["poly_open"]), 1))]) @ Hm.T
    if np.any(pts[:, 2] <= 1e-6):                       # 연장한 점이 지평선 너머로 넘어감 → 실패
        return None
    m = np.zeros((H, W), np.uint8)
    cv2.fillPoly(m, [np.round(pts[:, :2] / pts[:, 2:]).astype(np.int32).reshape(-1, 1, 2)], 1)
    return m


def mask_to_poly(m):
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return cv2.approxPolyDP(max(cnts, key=cv2.contourArea), 2.0, True).reshape(-1, 2)


def auto_roi(img, refs, model="homography", exclude=()):
    """→ (ROI 다각형 또는 None, 고른 참조 이름, 인라이어 수)"""
    H, W = img.shape[:2]
    pts, des = sift_features(img)
    best = None
    for ref in refs:
        if ref["name"] in exclude:
            continue
        Hm, n = estimate_transform(ref, pts, des, model)
        if Hm is not None and n >= MIN_INLIERS and (best is None or n > best[2]):
            best = (ref, Hm, n)
    if best is None:
        return None, None, 0
    m = warp_roi_mask(best[0], best[1], W, H)
    if m is None or m.sum() < 0.05 * W * H:             # 화면의 5%도 안 되면 잘못 옮긴 것으로 봄
        return None, best[0]["name"], best[2]
    return mask_to_poly(m), best[0]["name"], best[2]


def expand_polygon(poly, margin, W, H):
    """다각형을 margin(px)만큼 바깥으로 넓힘 (1a와 같은 방법: 채운 마스크 팽창 → 다시 다각형)"""
    mask = np.zeros((H, W), np.uint8)
    cv2.fillPoly(mask, [np.asarray(poly, np.int32).reshape(-1, 1, 2)], 255)
    if margin > 0:
        mask = cv2.dilate(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * margin + 1, 2 * margin + 1)))
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return cv2.approxPolyDP(max(cnts, key=cv2.contourArea), 2.0, True).reshape(-1, 2)


def poly_iou(p1, p2, W, H):
    a = np.zeros((H, W), np.uint8); b = np.zeros((H, W), np.uint8)
    cv2.fillPoly(a, [np.int32(p1).reshape(-1, 1, 2)], 1); cv2.fillPoly(b, [np.int32(p2).reshape(-1, 1, 2)], 1)
    return float((a & b).sum() / max(1, (a | b).sum()))

print("✅ 자동 ROI 함수 준비 완료 (OpenCV", cv2.__version__, ")")

In [ ]:
# ① 설정 · 라벨 (7a 그림을 보고 정한 값)
LINE_DIR = os.path.join(EVAL_DIR, "field_lines")
V7_DIR = os.path.join(DRIVE_OUTPUT_DIR, "v7")
FIELD_DIR = os.path.join(DRIVE_OUTPUT_DIR, "v7_field"); os.makedirs(FIELD_DIR, exist_ok=True)
DETECTED = json.load(open(os.path.join(LINE_DIR, "lines_detected.json"), encoding="utf-8"))

END_LINE = -10                 # 엔드존1(엔드라인) = 골라인 뒤 10야드 (엔드존 깊이 표준값을 선 기준 값으로 사용)
# 폭 방향 기준값 (야드). 대학 규칙: 사이드라인에서 해쉬까지 20야드, 해쉬 사이 13.33야드, 폭 53.33야드
WIDTH_NOMINAL = {"사이드L": 0.0, "해쉬L": 20.0, "해쉬R": 33.33, "사이드R": 53.33}
FIELD_LENGTH = 100             # 양쪽 골라인 사이 (경기장 전체 좌표: A쪽 골라인 0 → B쪽 골라인 100)

# 라벨: 숫자 = 야드 선(골라인 0 기준, 엔드라인은 END_LINE), "사이드" / "해쉬L" / "해쉬R". 적지 않은 선은 무시
# end: 0144 · 0143은 한쪽 엔드존(A), 0156 · 0157은 반대편 엔드존(B)
LABELS = {
    "DJI_20250913133612_0144_D": {"end": "A", "V1": "사이드", "V6": "사이드", "V3": "해쉬L", "V4": "해쉬R",
                                  "H2": END_LINE, "H3": 0, "H5": 10, "H7": 20},
    "DJI_20250913135020_0156_D": {"end": "B", "V7": "사이드", "V3": "해쉬R", "V4": "해쉬L",
                                  "H8": END_LINE, "H5": 0, "H4": 10, "H1": 20},
    "DJI_20250913133530_0143_D": {"end": "A", "V1": "사이드", "V6": "사이드", "V3": "해쉬L", "V4": "해쉬R",
                                  "H2": END_LINE, "H3": 0, "H4": 10, "H6": 20, "H7": 30},
    "DJI_20250913135111_0157_D": {"end": "B", "V1": "사이드", "V8": "사이드", "V4": "해쉬R", "V5": "해쉬L",
                                  "H7": END_LINE, "H4": 0, "H3": 10, "H1": 20},
}
for stem, lab in LABELS.items():
    miss = [k for k in lab if k != "end" and k not in DETECTED[stem]["lines"]]
    assert not miss, f"{stem[-6:-2]}: 7a에서 찾지 못한 선 번호 {miss} (7a를 다시 돌렸다면 번호가 바뀌었을 수 있어요)"
print("✅ 라벨 확인:", {s[-6:-2]: len(l) - 1 for s, l in LABELS.items()}, "개 선")

In [ ]:
# ② 참조 화면마다 보정
from scipy.optimize import least_squares

def line_of(d):
    return np.array(d["p1"], float), np.array(d["p2"], float)


def intersect(l1, l2):
    """두 직선(각각 두 점)의 교점 (무한히 늘린 직선 기준)"""
    (a1, a2), (b1, b2) = l1, l2
    A = np.array([a2 - a1, b1 - b2]).T
    if abs(np.linalg.det(A)) < 1e-9:
        return None
    t = np.linalg.solve(A, b1 - a1)[0]
    return a1 + t * (a2 - a1)


def mid(l):
    return (l[0] + l[1]) / 2


def width_lines(stem):
    """사이드라인을 해쉬L 쪽/해쉬R 쪽으로 자동 구분 → {"사이드L": line, "해쉬L": line, ...}"""
    lines, lab = DETECTED[stem]["lines"], LABELS[stem]
    out = {v: line_of(lines[k]) for k, v in lab.items() if v in ("해쉬L", "해쉬R")}
    for k, v in lab.items():
        if v == "사이드":
            l = line_of(lines[k])
            dL = np.linalg.norm(mid(l) - mid(out["해쉬L"])); dR = np.linalg.norm(mid(l) - mid(out["해쉬R"]))
            out["사이드L" if dL < dR else "사이드R"] = l
    return out


def apply_h(G, pts):
    pts = np.asarray(pts, float).reshape(-1, 2)
    q = np.hstack([pts, np.ones((len(pts), 1))]) @ G.T
    return q[:, :2] / q[:, 2:]


def samples(l, n=7):
    return [l[0] + (l[1] - l[0]) * t for t in np.linspace(0, 1, n)]


NORM = 1000.0          # 화면 좌표를 1000px 단위로 나눠 계산 (숫자 크기를 맞춰 최적화를 안정적으로)


def calibrate(stem):
    """화면 → 반듯한 격자 호모그래피를 찾고, 선마다 라벨 값을 붙이는 보정표를 만듦.
    1) 호모그래피는 '야드 선은 가로로, 폭 선은 세로로 곧게'만 맞춤. 양 끝 선 2개씩만 라벨 값에 고정하고
       가운데 선들의 위치는 자유롭게 둠 → 실제로 그려진 간격이 기준값과 달라도 원근이 비틀리지 않음
    2) 그렇게 찾은 각 선의 위치 → 라벨 값으로 선 사이를 비율(구간 선형)로 맞춤"""
    lines, lab = DETECTED[stem]["lines"], LABELS[stem]
    ylines = {v: line_of(lines[k]) for k, v in lab.items() if isinstance(v, (int, float))}
    wlines = width_lines(stem)
    assert len(ylines) >= 2 and len(wlines) >= 2, f"{stem[-6:-2]}: 야드 선 · 폭 선이 각각 2개 이상 필요해요"
    # 초기값: 교차점을 기준값에 맞춘 호모그래피 (이전 방식)
    img_pts, fld_pts = [], []
    for yv, yl in ylines.items():
        for wn, wl in wlines.items():
            p = intersect(yl, wl)
            if p is not None:
                img_pts.append(p / NORM); fld_pts.append((WIDTH_NOMINAL[wn], yv))
    G0, _ = cv2.findHomography(np.array(img_pts), np.array(fld_pts), 0)
    err_nominal = float(np.abs(apply_h(G0, img_pts) - np.array(fld_pts)).max())
    yk = sorted(ylines); wk = sorted(wlines, key=lambda k: WIDTH_NOMINAL[k])
    free_y, free_w = yk[1:-1], wk[1:-1]                                  # 양 끝 선은 라벨 값에 고정, 가운데는 자유
    sy = [(v, p / NORM) for v, l in ylines.items() for p in samples(l)]
    sw = [(k, p / NORM) for k, l in wlines.items() for p in samples(l)]

    def unpack(x):
        G = np.append(x[:8], 1.0).reshape(3, 3)
        ypos = {yk[0]: yk[0], yk[-1]: yk[-1], **dict(zip(free_y, x[8:8 + len(free_y)]))}
        wpos = {wk[0]: WIDTH_NOMINAL[wk[0]], wk[-1]: WIDTH_NOMINAL[wk[-1]], **dict(zip(free_w, x[8 + len(free_y):]))}
        return G, ypos, wpos

    def resid(x):
        G, ypos, wpos = unpack(x)
        qy = apply_h(G, [p for _, p in sy])[:, 1] - [ypos[v] for v, _ in sy]
        qw = apply_h(G, [p for _, p in sw])[:, 0] - [wpos[k] for k, _ in sw]
        return np.concatenate([qy, qw])

    x0 = np.concatenate([(G0 / G0[2, 2]).ravel()[:8], [float(v) for v in free_y], [WIDTH_NOMINAL[k] for k in free_w]])
    sol = least_squares(resid, x0, x_scale="jac")
    G, ypos, wpos = unpack(sol.x)
    cal = {"G": G, "end": lab["end"], "ylines": ylines, "wlines": wlines,
           "corr_y": sorted((ypos[v], v) for v in yk), "corr_w": sorted((wpos[k], WIDTH_NOMINAL[k]) for k in wk),
           "fit_y": {v: float(ypos[v]) for v in yk}, "fit_w": {k: float(wpos[k]) for k in wk},
           "err_nominal": err_nominal, "straight_rms": float(np.sqrt(np.mean(sol.fun ** 2))), "straight_max": float(np.abs(sol.fun).max())}
    return cal


def piecewise(x, table):
    """(호모그래피 값 → 라벨 값) 표로 구간 선형 보간. 표 밖은 가장 가까운 구간을 그대로 연장"""
    xs = np.array([t[0] for t in table]); vs = np.array([t[1] for t in table])
    if len(xs) == 1:
        return x - xs[0] + vs[0]
    i = np.clip(np.searchsorted(xs, x) - 1, 0, len(xs) - 2)
    return vs[i] + (x - xs[i]) * (vs[i + 1] - vs[i]) / (xs[i + 1] - xs[i])


def to_yards(cal, pts):
    """화면 점 → (폭 야드, 골라인 기준 야드) : 호모그래피로 원근을 편 뒤, 선 사이 비율로 맞춤"""
    m = apply_h(cal["G"], np.asarray(pts, float).reshape(-1, 2) / NORM)
    return np.stack([piecewise(m[:, 0], cal["corr_w"]), piecewise(m[:, 1], cal["corr_y"])], axis=1)


CAL = {}
rows = []
for stem in LABELS:
    c = calibrate(stem); CAL[stem] = c
    rows.append({"clip": stem[-6:-2], "엔드존": c["end"],
                 "그려진 야드 선 위치 (양 끝 고정)": ", ".join(f"{v:g}→{c['fit_y'][v]:.1f}" for v in sorted(c["fit_y"])),
                 "그려진 폭 선 위치 (양 끝 고정)": ", ".join(f"{k}={c['fit_w'][k]:.1f}" for k in sorted(c["fit_w"], key=lambda k: WIDTH_NOMINAL[k])),
                 "선 곧음 오차 평균 / 최대 (야드)": f"{c['straight_rms']:.2f} / {c['straight_max']:.2f}",
                 "(참고) 기준값 강제 시 오차": round(c["err_nominal"], 2)})
display(pd.DataFrame(rows))
print("- 그려진 위치: 양 끝 선을 라벨 값에 고정했을 때 가운데 선이 실제로 그려진 위치. 라벨 값과 다르면 그 경기장의 선 간격이 기준과 다른 것")
print("  (이 차이는 선 사이 비율 보정으로 흡수돼요: 그 선 위에 선 선수는 항상 라벨 값으로 계산됨)")
print("- 선 곧음 오차: 펴진 그림에서 선이 얼마나 곧은지. 0.3야드 이하면 좋음. 크면 렌즈 왜곡이나 잘못 찾은 선을 의심")

In [ ]:
# ③ 검증: (왼쪽) 보정으로 다시 그린 선 vs 실제 화면 · (오른쪽) 위에서 본 그림으로 펴기
def from_yards(cal, wy):
    """(폭, 야드) → 화면 점 (보정표를 거꾸로, 그다음 호모그래피 역변환)"""
    wy = np.asarray(wy, float).reshape(-1, 2)
    def inv_pw(v, t):
        xs = np.array([b for a, b in t]); ms = np.array([a for a, b in t])
        i = np.clip(np.searchsorted(xs, v) - 1, 0, len(xs) - 2)
        return ms[i] + (v - xs[i]) * (ms[i + 1] - ms[i]) / (xs[i + 1] - xs[i])
    m = np.stack([inv_pw(wy[:, 0], cal["corr_w"]), inv_pw(wy[:, 1], cal["corr_y"])], axis=1)
    return apply_h(np.linalg.inv(cal["G"]), m) * NORM


PX_PER_YD = 12
for stem, (fid, roi) in KNOWN.items():
    cal = CAL[stem]
    img = read_frames_sequential(video_path_of(stem), [fid])[fid]
    vis = img.copy()
    ys = np.arange(END_LINE, max(cal["ylines"]) + 11, 5)
    for yv in ys:                                                       # 5야드마다 가로선 (라벨 있는 선은 굵게)
        p = from_yards(cal, [[0, yv], [53.33, yv]]).astype(int)
        cv2.line(vis, tuple(p[0]), tuple(p[1]), (0, 255, 255) if yv in cal["ylines"] else (0, 160, 255), 3 if yv in cal["ylines"] else 1)
        cv2.putText(vis, f"{yv:g}", tuple(from_yards(cal, [[33.33 + 3, yv]])[0].astype(int)), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)
    for wn, wv in WIDTH_NOMINAL.items():
        p = from_yards(cal, [[wv, END_LINE], [wv, max(cal["ylines"]) + 10]]).astype(int)
        cv2.line(vis, tuple(p[0]), tuple(p[1]), (255, 255, 0), 2)
    # 위에서 본 그림: 폭 -5~58.3야드, 길이 END_LINE-5 ~ 마지막 선+10
    y0, y1 = END_LINE - 5, max(cal["ylines"]) + 10
    Wt, Ht = int(63.33 * PX_PER_YD), int((y1 - y0) * PX_PER_YD)
    S = np.array([[PX_PER_YD, 0, 5 * PX_PER_YD], [0, PX_PER_YD, -y0 * PX_PER_YD], [0, 0, 1]], float)
    N = np.diag([1 / NORM, 1 / NORM, 1.0])
    top = cv2.warpPerspective(img, S @ cal["G"] @ N, (Wt, Ht))          # 호모그래피만으로 편 그림 (선 사이 보정 전)
    fig, ax = plt.subplots(1, 2, figsize=(22, 8), gridspec_kw={"width_ratios": [2.2, 1]})
    ax[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); ax[0].axis("off")
    ax[0].set_title(f"{stem[-6:-2]} · 노랑 = 라벨 붙인 야드 선을 보정으로 다시 그림 · 주황 = 5야드 간격 · 하늘색 = 사이드 · 해쉬", fontsize=12)
    ax[1].imshow(cv2.cvtColor(top, cv2.COLOR_BGR2RGB)); ax[1].set_title("위에서 본 그림 (노란 격자 = 라벨 붙인 선이 있어야 할 자리)", fontsize=12)
    ax[1].set_yticks([(cal["fit_y"][v] - y0) * PX_PER_YD for v in sorted(cal["fit_y"])]); ax[1].set_yticklabels([f"{v:g}야드" for v in sorted(cal["fit_y"])])
    ax[1].set_xticks([(cal["fit_w"][k] + 5) * PX_PER_YD for k in cal["fit_w"]]); ax[1].set_xticklabels(list(cal["fit_w"]), fontsize=8)
    ax[1].grid(color="yellow", alpha=0.5)
    plt.tight_layout(); plt.show()
print("확인할 것: (왼쪽) 노랑 · 하늘색 선이 실제 흰 선 위에 겹치는지 · (오른쪽) 흰 선이 노란 격자와 나란히 반듯한지")

In [ ]:
# ④ v7 결과 → 매 프레임 선수 발 위치를 야드로
REFS = [make_ref(s, read_frames_sequential(video_path_of(s), [f])[f], roi) for s, (f, roi) in KNOWN.items()]
REF_BY_NAME = {r["name"]: r for r in REFS}


def frame_calibration(img):
    """이 프레임 → 가장 잘 맞는 참조 → (참조 보정, 화면→참조 호모그래피, 참조 이름, 인라이어)"""
    pts, des = sift_features(img)
    best = None
    for ref in REFS:
        Hm, n = estimate_transform(ref, pts, des)                       # 참조 → 이 프레임
        if Hm is not None and n >= MIN_INLIERS and (best is None or n > best[2]):
            best = (ref["name"], Hm, n)
    if best is None:
        return None
    name, Hm, n = best
    return {"ref": name, "H_to_ref": np.linalg.inv(Hm), "inliers": n}


def to_field(stem_ref, H_to_ref, feet):
    """화면 발 위치 → 참조 화면 → (폭 야드, 골라인 기준 야드, 경기장 전체 야드 0~100)"""
    cal = CAL[stem_ref]
    wy = to_yards(cal, apply_h(H_to_ref, feet))
    full = wy[:, 1] if cal["end"] == "A" else FIELD_LENGTH - wy[:, 1]
    return wy[:, 0], wy[:, 1], full


def convert_clip(stem):
    jp = os.path.join(V7_DIR, f"{stem}_tracking_final.json")
    data = json.load(open(jp, encoding="utf-8"))
    keys = sorted({1} | {int(k) for k in data.get("field_polygon_by_frame", {})})
    imgs = read_frames_sequential(video_path_of(stem), keys)
    calib, fails = {}, 0
    for k in keys:
        c = frame_calibration(imgs[k])
        if c is None:
            fails += 1
        else:
            calib[k] = c
    ok_keys = sorted(calib)
    n_pts = 0
    for fr in data["frames"]:
        prev = [k for k in ok_keys if k <= fr["frame_id"]]
        c = calib[prev[-1]] if prev else (calib[ok_keys[0]] if ok_keys else None)
        if c is None or not fr["players"]:
            continue
        w, y, full = to_field(c["ref"], c["H_to_ref"], [p["feet"] for p in fr["players"]])
        for p, a, b, f in zip(fr["players"], w, y, full):
            p["field"] = {"width_yd": round(float(a), 2), "yd_from_goal": round(float(b), 2), "end": CAL[c["ref"]]["end"],
                          "yd_100": round(float(f), 2)}
            n_pts += 1
    data["field_calibration"] = {str(k): {"ref": v["ref"][-6:-2], "inliers": v["inliers"]} for k, v in calib.items()}
    data["notes"].append("field: 선 기준 야드 (7b). width_yd = 해쉬L 쪽 사이드라인 0 · 해쉬L 20 · 해쉬R 33.33 · 반대 사이드 53.33, "
                         "yd_from_goal = 그 엔드존 골라인 0 · 엔드라인 -10, yd_100 = A쪽 골라인 0 → B쪽 골라인 100")
    out = os.path.join(FIELD_DIR, f"{stem}_tracking_field.json")
    json.dump(data, open(out, "w", encoding="utf-8"), ensure_ascii=False)
    refs = Counter(v["ref"][-6:-2] for v in calib.values())
    print(f"{stem[-6:-2]}: 보정 프레임 {len(calib)}/{len(keys)} (실패 {fails}) · 사용한 참조 {dict(refs)} · 야드로 바꾼 박스 {n_pts:,}개 → {out}")
    return data


FIELD = {stem: convert_clip(stem) for stem in KNOWN}

In [ ]:
# ⑤ 활용 1: 득점자 경로를 야드로 (6a에서 만든 득점자 정답 위치로 v7의 득점자 ID를 찾음)
SCORER_GT = json.load(open(os.path.join(EVAL_DIR, "scorer_gt_v2.json"), encoding="utf-8"))["gt"]
EVAL_STEMS = ["DJI_20250913133612_0144_D", "DJI_20250913135020_0156_D"]
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, stem in zip(axes, EVAL_STEMS):
    by_f = {fr["frame_id"]: fr["players"] for fr in FIELD[stem]["frames"]}
    ids = []
    for f, pos in SCORER_GT[stem].items():
        if pos is None:
            continue
        near = [(np.hypot(p["feet"][0] - pos[0], p["feet"][1] - pos[1]), p["track_id"]) for p in by_f.get(int(f), [])]
        if near and min(near)[0] < 30:
            ids.append(min(near)[1])
    sid = Counter(ids).most_common(1)[0][0]
    path = [(f, p["field"]) for f, ps in sorted(by_f.items()) for p in ps if p["track_id"] == sid and "field" in p]
    fr_ids = [f for f, _ in path]; w = np.array([d["width_yd"] for _, d in path]); y = np.array([d["yd_from_goal"] for _, d in path])
    ax.plot(w, y, "-", color="crimson", lw=2); ax.plot(w[0], y[0], "go", ms=10, label=f"시작 f{fr_ids[0]}"); ax.plot(w[-1], y[-1], "k*", ms=16, label=f"끝 f{fr_ids[-1]}")
    for v in range(END_LINE, 41, 5):
        ax.axhline(v, color="gray", lw=1.5 if v % 10 == 0 else 0.5, alpha=0.6)
    for v in WIDTH_NOMINAL.values():
        ax.axvline(v, color="steelblue", lw=1, alpha=0.6)
    ax.axhspan(END_LINE, 0, color="gold", alpha=0.15, label="엔드존")
    ax.set_xlim(-5, 58.33); ax.set_ylim(40, END_LINE - 3); ax.set_aspect("equal")
    ax.set_xlabel("폭 (야드, 해쉬L 쪽 사이드라인 = 0)"); ax.set_ylabel("골라인까지 야드 (0 = 골라인)")
    ax.set_title(f"{stem[-6:-2]} · 득점자 #{sid} 경로 ({len(path)}프레임) · 시작 {y[0]:.1f}야드 → 끝 {y[-1]:.1f}야드")
    ax.legend(loc="lower right"); ax.grid(False)
    print(f"{stem[-6:-2]} 득점자 #{sid}: 골라인까지 {y[0]:.1f} → {y[-1]:.1f}야드 (전진 {y[0] - y[-1]:.1f}야드) · 폭 {w[0]:.1f} → {w[-1]:.1f}")
plt.tight_layout(); plt.show()

In [ ]:
# ⑤ 활용 2: 경기장 밖(사이드라인 · 엔드라인 밖) 사람 걸러내면 한 팀 인원이 11명 근처로 오는지
OUT_MARGIN = 1.0      # 라인 밖 1야드까지는 경기장 안으로 봄 (발 위치 오차 여유)
summary = []
for stem, data in FIELD.items():
    before, after = [], []
    for fr in data["frames"]:
        ps = [p for p in fr["players"] if "field" in p]
        inside = [p for p in ps if -OUT_MARGIN <= p["field"]["width_yd"] <= 53.33 + OUT_MARGIN and p["field"]["yd_from_goal"] >= END_LINE - OUT_MARGIN]
        cb = Counter(p["type"] for p in ps); ca = Counter(p["type"] for p in inside)
        before.append(max(cb["Hongik"], cb["Kookmin"])); after.append(max(ca["Hongik"], ca["Kookmin"]))
    before, after = np.array(before), np.array(after)
    summary.append({"clip": stem[-6:-2], "14명 초과 프레임 (v7)": int((before > 14).sum()), "14명 초과 프레임 (라인 밖 제외)": int((after > 14).sum()),
                    "더 많은 팀 평균 (v7)": round(before.mean(), 1), "더 많은 팀 평균 (라인 밖 제외)": round(after.mean(), 1)})
display(pd.DataFrame(summary))
print("정상은 팀당 11명. 이건 확인용 계산이고 아직 v7 결과 파일은 바꾸지 않아요.")

In [ ]:
# ⑥ 한 팀이 14명을 넘는 프레임 눈으로 보기: 교체 · 득점 후 축하처럼 실제로 경기장에 많은 건지, 잘못 잡은 건지
TEAM_BGR = {"Hongik": (255, 140, 0), "Kookmin": (40, 40, 230), "Referee": (0, 215, 255)}
for stem in EVAL_STEMS:
    frs = FIELD[stem]["frames"]
    by_f = {fr["frame_id"]: fr["players"] for fr in frs}
    over = [f for f, ps in by_f.items() if max(Counter(p["type"] for p in ps)[t] for t in ("Hongik", "Kookmin")) > 14]
    if not over:
        print(f"{stem[-6:-2]}: 14명 초과 프레임 없음"); continue
    runs, start = [], over[0]                                          # 연속 구간으로 묶기
    for a, b in zip(over, over[1:] + [None]):
        if b is None or b - a > 5:
            runs.append((start, a)); start = b
    print(f"{stem[-6:-2]}: 14명 초과 {len(over)}프레임 · 구간 {runs}")
    pick = sorted({over[0], over[len(over) // 2], over[-1]})
    imgs = read_frames_sequential(video_path_of(stem), pick)
    fig, axes = plt.subplots(1, len(pick), figsize=(8 * len(pick), 5), squeeze=False)
    for ax, f in zip(axes[0], pick):
        im = imgs[f].copy(); cnt = Counter(p["type"] for p in by_f[f])
        for p in by_f[f]:
            x1, y1, x2, y2 = p["bbox"]
            cv2.rectangle(im, (x1, y1), (x2, y2), TEAM_BGR.get(p["type"], (160, 160, 160)), 3)
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.axis("off")
        ax.set_title(f"{stem[-6:-2]} · f{f} · 홍익(파랑) {cnt['Hongik']} · 국민(빨강) {cnt['Kookmin']} · 심판(노랑) {cnt['Referee']}", fontsize=12)
    plt.tight_layout(); plt.show()
print("확인할 것: 실제로 경기장 안에 그 팀 사람이 많은지(교체 중, 득점 후 몰려나옴 등), 아니면 한 사람에게 박스가 겹쳐 있거나 다른 팀을 잘못 칠했는지")

In [ ]:
# ⑦ 경기 기록과 맞춰보기: LOS(공 위치) · 터치다운 순간 · 전진 야드
# 기록(경기 기록 엑셀): 두 득점 모두 상대 3야드에서 RUN, 전진 3야드, TOUCHDOWN
# 전진 야드 규칙: LOS → 0야드(골라인)까지만 인정. 골라인을 넘은 뒤 움직임은 세지 않음
PLAY_LOG = {
    "DJI_20250913133612_0144_D": {"clipkey": 34, "offense": "Hongik", "carrier": "RB #25", "los": 3, "gain": 3, "result": "TOUCHDOWN", "play_start": 413},
    "DJI_20250913135020_0156_D": {"clipkey": 44, "offense": "Kookmin", "carrier": "RB #26", "los": 3, "gain": 3, "result": "TOUCHDOWN", "play_start": 167},
}
LOS_LINE = {}    # 그려진 LOS 표시 선이 있는 경기장이면 {클립: "선 번호"}로 적어 비교 (0144의 H4는 선수들이 일렬로 선 모습을 선으로 잘못 찾은 것이라 뺌)


def field_players(ps):
    return [p for p in ps if "field" in p and p["type"] in ("Hongik", "Kookmin")
            and -1 <= p["field"]["width_yd"] <= 54.33 and p["field"]["yd_from_goal"] >= END_LINE - 1]


def estimate_los(players, offense, k=5):
    """플레이 시작 순간: 공격 팀에서 골라인에 가장 가까운 k명(라인맨)과 수비 팀에서 골라인에서 가장 먼 k명(수비 앞줄)의 가운데"""
    off = sorted(p["field"]["yd_from_goal"] for p in players if p["type"] == offense)
    dfn = sorted(p["field"]["yd_from_goal"] for p in players if p["type"] != offense)
    if len(off) < k or len(dfn) < k:
        return None
    return (np.median(off[:k]) + np.median(dfn[-k:])) / 2


rows = []
for stem, log in PLAY_LOG.items():
    by_f = {fr["frame_id"]: fr["players"] for fr in FIELD[stem]["frames"]}
    f0 = log["play_start"]
    los_est = estimate_los(field_players(by_f.get(f0, [])), log["offense"])
    los_line = None
    if stem in LOS_LINE:
        l = DETECTED[stem]["lines"][LOS_LINE[stem]]
        ys = to_yards(CAL[stem], [l["p1"], l["p2"], (np.array(l["p1"]) + np.array(l["p2"])) / 2])[:, 1]
        los_line = f"{ys[2]:.1f} (양 끝 {ys[0]:.1f} / {ys[1]:.1f})"
    # 볼 캐리어 = 6a 득점자 정답 위치와 가장 가까운 트랙
    ids = []
    for f, pos in SCORER_GT[stem].items():
        if pos is None:
            continue
        near = [(np.hypot(p["feet"][0] - pos[0], p["feet"][1] - pos[1]), p["track_id"]) for p in by_f.get(int(f), [])]
        if near and min(near)[0] < 30:
            ids.append(min(near)[1])
    sid = Counter(ids).most_common(1)[0][0]
    path = [(f, p["field"]["yd_from_goal"]) for f in sorted(by_f) if f >= f0 for p in by_f[f] if p["track_id"] == sid and "field" in p]
    start_yd = path[0][1] if path else None
    td = next((f for f, y in path if y <= 0), None)
    end_yd = 0.0 if td else (path[-1][1] if path else None)          # 터치다운이면 골라인(0)에서 끝, 이후 움직임 무시
    gain = None if (los_est is None or end_yd is None) else los_est - max(0.0, end_yd)
    rows.append({"clip": stem[-6:-2], "ClipKey": log["clipkey"], "기록 LOS": log["los"], "그려진 LOS 선 계산": los_line,
                 "LOS 추정 (라인맨 사이)": None if los_est is None else round(los_est, 1),
                 "캐리어 트랙": f"#{sid}", "캐리어 시작 위치": None if start_yd is None else round(start_yd, 1),
                 "LOS 뒤 깊이": None if (start_yd is None or los_est is None) else round(start_yd - los_est, 1),
                 "골라인 통과 프레임": td, "기록 결과": log["result"], "기록 전진": log["gain"],
                 "계산 전진 (LOS → 0까지만)": None if gain is None else round(gain, 1)})
display(pd.DataFrame(rows).set_index("clip").T)
print("맞춰볼 것: LOS 추정 ≈ 기록 LOS(3) · 골라인 통과 프레임이 있음(= 터치다운) · 계산 전진 ≈ 기록 전진(3)")
print("캐리어 시작 위치는 공이 아니라 RB가 선 자리라 LOS보다 몇 야드 뒤가 정상이에요.")